In [ ]:
import duckdb
from pathlib import Path

In [ ]:
partitioned = Path("../catalog/h3v2r1/")

In [ ]:
duckdb.sql(
    f"select column_name from (describe select * from read_parquet('{partitioned}/**/*.parquet'))"
)

In [ ]:
import json
import logging
import os
import math

import rustac
import duckdb
import pandas as pd
import s3fs

# from pqdm.threads import pqdm
from shapely.geometry import shape, box, mapping, Polygon
import math

logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO)



colombia = {
  "type": "Feature",
  "geometry": {
    "type": "Polygon",
    "coordinates": [
          [
            [
              -76.8118477368562,
              10.725951506531445
            ],
            [
              -76.8118477368562,
              4.325087420870503
            ],
            [
              -69.60861905318234,
              4.325087420870503
            ],
            [
              -69.60861905318234,
              10.725951506531445
            ],
            [
              -76.8118477368562,
              10.725951506531445
            ]
          ]
        ]
  },
  "properties": {}
}

hma = {
  "type": "Feature",
  "properties": {},
  "geometry": {
    "type": "Polygon",
    "coordinates": [
      [
        [83.221436, 31.503629],
        [84.968262, 31.503629],
        [84.968262, 32.833443],
        [83.221436, 32.833443],
        [83.221436, 31.503629]
      ]
    ]
  }
}

greenland = {
  "type": "Feature",
  "properties": {},
  "geometry": {
    "type": "Polygon",
    "coordinates": [
    [
        [-45.05, 61.93527564358849],
        [-44.537947278569234, 61.93439779524844],
        [-44.07595464722729, 61.931764440940775],
        [-43.61408214918781, 61.927376152621925],
        [-43.1523897340054, 61.92123388294972],
        [-43.13737905069332, 62.13891672639419],
        [-43.122122552714636, 62.35679581675177],
        [-43.10661415425373, 62.57486983167596],
        [-43.09084756700362, 62.793137439704125],
        [-43.56790381583535, 62.79950724447399],
        [-44.04515874612781, 62.804058153589295],
        [-44.52254622269042, 62.80678911869741],
        [-45.05, 62.80769951036124],
        [-45.05, 62.5892987291677],
        [-45.05, 62.37109363832143],
        [-45.05, 62.153085518672796],
        [-45.05, 61.93527564358849]]
    ]
  }
}

In [ ]:
import h3
h3_level = 2
geojson_geometry = greenland["geometry"]
grids_hex = h3.h3shape_to_cells_experimental(h3.geo_to_h3shape(geojson_geometry), 1, "bbox_overlap")
grids = [int(hs, 16) for hs in grids_hex]
grids

In [ ]:
def lat_prefix(lat):
    return f"N{abs(lat):02d}" if lat >= 0 else f"S{abs(lat):02d}"

def lon_prefix(lon):
    return f"E{abs(lon):03d}" if lon >= 0 else f"W{abs(lon):03d}"


def get_overlapping_h3_names(geojson_geometry, level: int = 2, contain: str = "overlap"):
    """
    This is a workaround, the spatial partitioning should be in the parquet files
    """
    import h3
    grids_hex = h3.h3shape_to_cells_experimental(h3.geo_to_h3shape(geojson_geometry), level, contain)
    grids = [int(hs, 16) for hs in grids_hex]
    return grids


def get_overlapping_grid_names(geojson_geometry):
    """
    This is a workaround, the spatial partitioning should be in the parquet files
    """
    geom = shape(geojson_geometry)
    if not geom.is_valid:
        geom = geom.buffer(0)

    minx, miny, maxx, maxy = geom.bounds

    # Center-based grid
    lon_center_start = int(math.floor((minx - 5) / 10.0)) * 10
    lon_center_end   = int(math.ceil((maxx + 5) / 10.0)) * 10
    lat_center_start = int(math.floor((miny - 5) / 10.0)) * 10
    lat_center_end   = int(math.ceil((maxy + 5) / 10.0)) * 10
    
    result = set()
    for lon_c in range(lon_center_start, lon_center_end + 1, 10):
        for lat_c in range(lat_center_start, lat_center_end + 1, 10):
            tile = box(lon_c - 5, lat_c - 5, lon_c + 5, lat_c + 5)
            if geom.intersects(tile):
                name = f"{lat_prefix(lat_c)}{lon_prefix(lon_c)}"
                result.add(name)
    return sorted(result)


def find_paths_for_dir_names(base_dir, dir_names):
    base = Path(base_dir)
    matches = []

    for dir_name in dir_names:
        matches.extend(p for p in base.rglob(dir_name) if p.is_dir())
    return matches

def path_exists(path: str) -> bool:
    if path.startswith("s3://"):
        fs = s3fs.S3FileSystem(anon=True)
        return fs.exists(path)
    else:
        return os.path.exists(path)

def build_cql2_filter(filters_list):
    if not filters_list:
        return None
    return filters_list[0] if len(filters_list) == 1 else {"op": "and", "args": filters_list}


def serverless_search(base_catalog_href: str = "s3://its-live-data/test-space/stac",
                      search_kwargs: dict = [],
                      reduce_spatial_search=True,
                      partitioning: str = "h3",
                      partitioning_level: int = 2,
                      contain: str = "overlap",
                      asset_type: str = ".nc"):
    """
    base_catalog_href: uri for the ITS_LIVE STAC catalog e.g. "s3://its-live-data/test-space/stac"
    search_kwargs: STAC api search kwargs.

    Returns a list of URL matching the asset type (string matching not mime type)
    """
    client = rustac.DuckdbClient()
    backend = base_catalog_href
    missions = ["landsatOLI", "sentinel1", "sentinel2"]
    if reduce_spatial_search:
        if "intersects" in search_kwargs:
            if partitioning == "h3":
                grids = get_overlapping_h3_names(search_kwargs["intersects"], level=partitioning_level, contain=contain)
                prefixes = [f"{backend}/{p}" for p in grids]
                search_prefixes = [f"{prefix}/**/*.parquet" for prefix in prefixes if path_exists(prefix)]
            elif partitioning == "latlon":
                grids = get_overlapping_grid_names(search_kwargs["intersects"])
                prefixes = [f"{backend}/{p}/{i}" for p in missions for i in grids]
                search_prefixes = [f"{path}/**/*.parquet" for path in prefixes if path_exists(path)]
    else:
        search_prefixes = [f"{backend}/**/*.parquet"]

    print((f"Searching in {search_prefixes}"))
    
    hrefs = []
    for prefix in search_prefixes:
        try:
            if search_kwargs:
                try:
                    items = client.search(prefix, **search_kwargs)
                except Exception as e:
                    print(f"Error querying: {prefix}, {e}")
                
            else:
                items = client.search(prefix, collection=["itslive-granules"])
            for item in items:
                for asset in item["assets"].values():
                    if "data" in asset["roles"] and asset["href"].endswith(".nc"):
                        hrefs.append(asset["href"])            
            print(f"Prefx: {prefix} items found: {len(items)}")
        except Exception as e:
            print(f"Error while searching in {prefix}: {e}")
        
    return hrefs

In [ ]:
%%time

filters = [
    {"op": ">=", "args": [{"property": "percent_valid_pixels"}, 1]},
    {'op': '=', 'args': [{'property': 'proj:code'}, 'EPSG:3413']}
]

search_kwargs = {
    "intersects": greenland["geometry"], # <- has to be in lat lon 
    # "datetime": "1980-01-01T00:00:00Z/2025-12-31T23:59:59Z",
    "filter": build_cql2_filter(filters)
}

catalog_base_href = "s3://its-live-data/test-space/stac/geoparquet/h3r1"

# catalog_base_href = "s3://its-live-data/test-space/stac"
# catalog_base_href = "../catalog/h3v2"
results = serverless_search(catalog_base_href,
                            search_kwargs=search_kwargs,
                            reduce_spatial_search=True,
                            partitioning="h3",
                            partitioning_level=1,
                            contain="overlap" # contain ({'center', 'full', 'overlap', 'bbox_overlap'}, optional) – 
                           )

len(results)

In [ ]:


# Connect to DuckDB
con = duckdb.connect()

# Load spatial extension
con.execute("INSTALL spatial")
con.execute("LOAD spatial")


def expr_to_sql(expr):
    op = expr["op"]
    left, right = expr["args"]
    
    # Get property name if dict with "property" key, else literal
    def val_to_sql(val):
        if isinstance(val, dict) and "property" in val:
            # Quote identifiers with special chars
            prop = val["property"]
            if not prop.isidentifier():
                return f'"{prop}"'
            return prop
        elif isinstance(val, str):
            # quote strings
            return f"'{val}'"
        else:
            # numbers or others
            return str(val)

    left_sql = val_to_sql(left)
    right_sql = val_to_sql(right)

    # Map operators (expand as needed)
    op_map = {
        "=": "=",
        "==": "=",
        ">=": ">=",
        "<=": "<=",
        ">": ">",
        "<": "<",
        "!=": "<>",
        "<>": "<>"
    }
    sql_op = op_map.get(op, op)

    return f"{left_sql} {sql_op} {right_sql}"

def filters_to_where(filters):
    # filters is a list of expressions combined with AND
    sql_parts = [expr_to_sql(f) for f in filters]
    return " AND ".join(sql_parts)

def duck_search(catalog,
                geometry,
                reduce_spatial_search=True, 
                partitioning="latlon",
                partitioning_level: int = 2,
                contain: str = "overlap",
                filters: list = []):
    geojson_str = json.dumps(geometry)
    missions = ["landsatOLI", "sentinel1", "sentinel2"]

    if reduce_spatial_search:
        if "intersects" in search_kwargs:
            if partitioning == "h3":
                grids = get_overlapping_h3_names(search_kwargs["intersects"], partitioning_level, contain)
                prefixes = [f"{catalog}/{p}" for p in grids]
                search_prefixes = [f"{prefix}/**/*.parquet" for prefix in prefixes if path_exists(prefix)]
            elif partitioning == "latlon":
                grids = get_overlapping_grid_names(search_kwargs["intersects"])
                prefixes = [f"{catalog}/{p}/{i}" for p in missions for i in grids]
                search_prefixes = [f"{path}/**/*.parquet" for path in prefixes if path_exists(path)]
    else:
        search_prefixes = [f"{catalog}/{path}/**/*.parquet" for path in missions]
        
    print(f"Searching in: {search_prefixes}")    
    # search_prefixes = glob.glob("geoparquet/**/*.parquet", recursive=True)

    results = []
    filters_sql = filters_to_where(filters)
    print(f"Filters as SQL: {filters_sql}")
    
    for path in search_prefixes:
        query = f"""
            SELECT 
                '{path}' AS source_parquet,
                assets -> 'data' ->> 'href' AS data_href
            FROM read_parquet('{path}')
            WHERE ST_Intersects(
                geometry,
                ST_GeomFromGeoJSON('{geojson_str}')
            ) AND {filters_sql}
        """
        df = con.execute(query).df()
        results.append(df)
    
    final_df = pd.concat(results, ignore_index=True)
    hrefs = final_df["data_href"].to_list() 
    return hrefs

In [ ]:
%%time
# catalog_url = "s3://its-live-data/test-space/stac/geoparquet/h3r1"
catalog_url = "../catalog/geoparquet/"

filters = [
    {"op": ">=", "args": [{"property": "percent_valid_pixels"}, 1]},
    {"op": "=", "args": [{"property": "proj:code"}, "EPSG:3413"]}
]

search_kwargs = {
    "intersects": greenland["geometry"], # <- has to be in lat lon 
    # "datetime": "1980-01-01T00:00:00Z/2025-12-31T23:59:59Z",
    "filter": build_cql2_filter(filters)
}

#not using time yet, will return all the time series
results = duck_search(catalog_url,
                      greenland["geometry"],
                      reduce_spatial_search=True,
                      partitioning="latlon",
                      partitioning_level=2,
                      contain="bbox_overlap",
                      filters=filters)

len(results)

In [ ]:
%%time
catalog_url = "s3://its-live-data/test-space/stac/geoparquet/h3r2"
# catalog_url = "geoparquet"

filters = [
    {"op": ">=", "args": [{"property": "percent_valid_pixels"}, 1]},
    {"op": "=", "args": [{"property": "proj:code"}, "EPSG:3413"]}
]

search_kwargs = {
    "intersects": greenland["geometry"], # <- has to be in lat lon 
    # "datetime": "1980-01-01T00:00:00Z/2025-12-31T23:59:59Z",
    "filter": build_cql2_filter(filters)
}

#not using time yet, will return all the time series
results = duck_search(catalog_url,
                      greenland["geometry"],
                      reduce_spatial_search=True,
                      partitioning="h3",
                      partitioning_level=2,
                      filters=filters)
len(results)

In [ ]:
results[0]